# Ohm's law from first principles: $\dfrac{dQ}{dt}\,R = V$

Ohm's law is usually just handed to you as $V=IR$. Current $I$ is, by
definition, the rate charge flows: $I \equiv \dfrac{dQ}{dt}$. Put those
together for a single-loop RC circuit and Ohm's law stops being a fact you
memorize and becomes a differential equation you can solve.

That solved equation turns out to be a formula this repo already uses in
two other places without ever deriving it:
[`dgs/memory_circuits.py`](../dgs/memory_circuits.py)'s `dram_cell_decay`
states $V(t)=V_0 e^{-t/(R_{\text{leak}}C_{\text{cell}})}$ as a given fact
("same RC decay as..."), and
[`dgs/molecular_ohms_law.py`](../dgs/molecular_ohms_law.py) drills $V=IR$
*downward* into the Drude model without ever justifying the $V=IR$ starting
point itself. This notebook derives it, then checks it against both.


In [1]:
import sys, pathlib
import numpy as np
import sympy as sp

sp.init_printing(use_latex="mathjax")

REPO = pathlib.Path(r"D:/Summer2026/Dispersion-Assisted-GS-Phase-Recovery")
sys.path.insert(0, str(REPO))
from dgs import memory_circuits as mc
from dgs import molecular_ohms_law as mol

checks = []


def check(label, condition):
    checks.append((label, bool(condition)))
    print(f"{'PASS' if condition else 'FAIL'}  —  {label}")


## 1. Set up the single-loop RC circuit

A capacitor $C$, charged to $Q_0$, discharges through a resistor $R$.
Charge is leaving the capacitor, so the current through the resistor is
$I=-\dfrac{dQ}{dt}$ (the minus sign is bookkeeping, not new physics — $Q$ is
decreasing, so $-dQ/dt>0$ is the actual current flowing). Kirchhoff's
voltage law says the resistor's voltage drop equals the capacitor's
voltage: $IR = Q/C$.


In [2]:
t, R, C, Q0 = sp.symbols("t R C Q0", positive=True)
Q = sp.Function("Q")

I_expr = -sp.Derivative(Q(t), t)
ode = sp.Eq(I_expr * R, Q(t) / C)

print("Ohm's law (V=IR) combined with I = -dQ/dt and V_C = Q/C:")
sp.pretty_print(ode)


Ohm's law (V=IR) combined with I = -dQ/dt and V_C = Q/C:
   d          Q(t)
-R⋅──(Q(t)) = ────
   dt          C  


In [3]:
solution = sp.dsolve(ode, Q(t), ics={Q(0): Q0})
Q_of_t = solution.rhs

print("Solving the ODE:")
sp.pretty_print(solution)

check("Solving dQ/dt*R = -Q/C gives exponential decay",
      Q_of_t.has(sp.exp) and sp.simplify(Q_of_t.subs(t, 0) - Q0) == 0)


Solving the ODE:
           -t 
           ───
           C⋅R
Q(t) = Q₀⋅ℯ   
PASS  —  Solving dQ/dt*R = -Q/C gives exponential decay


## 2. Cross-check against `dgs.memory_circuits.dram_cell_decay`

$V(t) = Q(t)/C$, and with $Q_0 = V_0 C$ this should reproduce
`dram_cell_decay`'s formula **exactly** — not approximately, since both
sides are solving the same linear ODE.


In [4]:
Q_fn = sp.lambdify((t, R, C, Q0), Q_of_t, "numpy")

V0 = 3.3
R_val, C_val = 3e12, 30e-15
times = np.array([0.0, 1e-3, 5e-3, 1e-2])

V_from_derivation = Q_fn(times, R_val, C_val, V0 * C_val) / C_val   # V = Q/C
V_from_module = mc.dram_cell_decay(V0, times, R_val, C_val)

print("t (s)         derived V(t)      dram_cell_decay V(t)")
for tt, a, b in zip(times, V_from_derivation, V_from_module):
    print(f"{tt:10.2e}   {a:14.8f}   {b:14.8f}")

max_diff = float(np.max(np.abs(V_from_derivation - V_from_module)))
print(f"\nmax|derived - dram_cell_decay| = {max_diff:.2e}")
check("First-principles derivation reproduces dram_cell_decay to machine precision", max_diff < 1e-9)


t (s)         derived V(t)      dram_cell_decay V(t)


  0.00e+00       3.30000000       3.30000000
  1.00e-03       3.26353628       3.26353628
  5.00e-03       3.12166625       3.12166625
  1.00e-02       2.95296975       2.95296975

max|derived - dram_cell_decay| = 4.44e-16
PASS  —  First-principles derivation reproduces dram_cell_decay to machine precision


## 3. The static limit: plain $V=IR$ falls out for free

If the current is constant ($I=I_0$, not a decaying capacitor), then
$Q(t)=Q_0+I_0 t$ is linear in $t$, and the resistor's instantaneous voltage
drop is just $I_0 R$ — the ordinary, time-independent Ohm's law. $dQ/dt\,R=V$
isn't a *different* law from $V=IR$; $V=IR$ is the special case where the
current happens to be constant.


In [5]:
I0, R_static = sp.symbols("I0 R", positive=True)
Q_linear = Q0 + I0 * t          # constant current -> charge grows linearly
I_from_Q = sp.diff(Q_linear, t)  # dQ/dt
V_static = I_from_Q * R_static

print(f"dQ/dt for constant current I0: {I_from_Q}")
print(f"V = (dQ/dt)*R = {V_static}   (independent of t, as expected)")

check("Constant current recovers the ordinary, time-independent V=I*R",
      sp.simplify(V_static - I0 * R_static) == 0 and not V_static.has(t))

# cross-check the NUMBER against molecular_ohms_law's own V=IR function
v_module = mol.ohms_law_voltage(2.0, 5.0)
v_derived = float(V_static.subs({I0: 2.0, R_static: 5.0}))
print(f"\nmolecular_ohms_law.ohms_law_voltage(2.0, 5.0) = {v_module}")
print(f"static limit of dQ/dt*R at I0=2.0, R=5.0        = {v_derived}")
check("Static limit matches molecular_ohms_law.ohms_law_voltage exactly", v_module == v_derived)


dQ/dt for constant current I0: I0
V = (dQ/dt)*R = I0*R   (independent of t, as expected)
PASS  —  Constant current recovers the ordinary, time-independent V=I*R

molecular_ohms_law.ohms_law_voltage(2.0, 5.0) = 10.0
static limit of dQ/dt*R at I0=2.0, R=5.0        = 10.0
PASS  —  Static limit matches molecular_ohms_law.ohms_law_voltage exactly


So the same statement — $\dfrac{dQ}{dt}R=V$ — is the RC discharge
curve `dram_cell_decay` already relies on when current is *changing*, and
plain $V=IR$ (the starting point `molecular_ohms_law.py` drills down into
the Drude model) when current is *constant*. Neither module needed to
re-derive this; it was already sitting one differential equation away from
the definition of current.

## Final grade

In [6]:
failures = [label for label, ok in checks if not ok]
print(f"{len(checks) - len(failures)}/{len(checks)} checks passed")

if failures:
    raise AssertionError("Failed checks: " + ", ".join(failures))
else:
    print("\nALL CHECKS PASSED — dQ/dt*R=V, solved from scratch, reproduces "
          "dgs.memory_circuits.dram_cell_decay exactly in the dynamic case and "
          "dgs.molecular_ohms_law.ohms_law_voltage exactly in the static limit.")


4/4 checks passed

ALL CHECKS PASSED — dQ/dt*R=V, solved from scratch, reproduces dgs.memory_circuits.dram_cell_decay exactly in the dynamic case and dgs.molecular_ohms_law.ohms_law_voltage exactly in the static limit.
